# Session 5: Design Patterns II — Behavioral

> Master Observer, Strategy, and Command patterns for flexible system interactions.

## 1. Observer Pattern

### Why Observer?
**"Define a one-to-many dependency so that when one object changes state, all its dependents are notified automatically."**

Without Observer, you would write: `send_email(order); update_warehouse(order); track_analytics(order)` every time an order is created. When a fourth team needs to add a push notification, they ask you to edit that function. With Observer, they simply subscribe to the `'order.created'` event — your code never changes.

**Event Bus vs direct Observer**
The `EventBus` below decouples publishers from subscribers completely — neither knows the other exists. The trade-off is that the event contract (the payload structure) must be documented carefully, or subscribers break silently when the payload changes.

**When to be careful:** Event-driven code is harder to trace — a bug in a subscriber does not show up at the publish call site. Always log event dispatches in production, and consider typed event payloads (dataclasses) so mismatches are caught at type-check time rather than at runtime.

In [ ]:
class EventBus:
    def __init__(self):
        self._handlers = {}
    def subscribe(self, event: str, handler):
        self._handlers.setdefault(event, []).append(handler)
    def publish(self, event: str, payload=None):
        for h in self._handlers.get(event, []):
            h(payload)

bus = EventBus()
bus.subscribe('order.created', lambda d: print(f'📧 Email: Order {d["id"]} confirmed'))
bus.subscribe('order.created', lambda d: print(f'📦 Warehouse: Pick order {d["id"]}'))
bus.subscribe('order.created', lambda d: print(f'📊 Analytics: Track conversion'))
bus.publish('order.created', {'id': 'ORD-001', 'amount': 150})

## 💡 Additional Examples: Observer, Command & Template Method

**Example 1 — Typed Observer: Stock Price Monitor**
Using a `dataclass` for the event payload (`PriceChangeEvent`) instead of a raw `dict` means subscribers get autocompletion and type checking. If the payload structure changes, the type checker catches all broken subscribers before runtime. This is the production-quality version of the EventBus shown above.

**Example 2 — Command Pattern with Undo/Redo**
The Command pattern wraps each user action as an object with `execute()` and `undo()`. The undo/redo stack is simply a `list` of Command objects — `undo()` pops from the done stack, and `redo()` moves it back. This is exactly how every text editor, drawing tool, and spreadsheet implements Ctrl+Z. The key insight: encapsulating actions as objects means you can queue them, log them, serialize them, and replay them.

**Example 3 — Template Method: Data Pipeline**
Template Method defines the *skeleton* of an algorithm in a base class and lets subclasses fill in specific steps. `DataPipeline.run()` always does: extract → transform → load → notify. `CSVToDatabase` and `JSONToElasticsearch` override only the steps that differ. This avoids copy-pasting the orchestration logic into every pipeline variant.

In [ ]:
# ─── Example 1: Observer with typed events — Stock Price Monitor ──────────────
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Callable

@dataclass
class PriceChangeEvent:
    symbol: str
    old_price: float
    new_price: float

    @property
    def change_pct(self): return (self.new_price - self.old_price) / self.old_price * 100

class StockMarket:
    def __init__(self):
        self._prices: dict[str, float] = {}
        self._observers: list[Callable[[PriceChangeEvent], None]] = []

    def subscribe(self, observer: Callable): self._observers.append(observer)

    def update_price(self, symbol: str, new_price: float):
        old = self._prices.get(symbol, new_price)
        self._prices[symbol] = new_price
        if old != new_price:
            event = PriceChangeEvent(symbol, old, new_price)
            for obs in self._observers:
                obs(event)
# Observers — each reacts independently to the same event
def alert_large_move(e: PriceChangeEvent):
    if abs(e.change_pct) >= 5:
        direction = '🔺' if e.change_pct > 0 else '🔻'
        print(f'  {direction} ALERT: {e.symbol} {e.change_pct:+.1f}%  ({e.old_price} → {e.new_price})')

def log_all_changes(e: PriceChangeEvent):
    print(f'  📊 LOG: {e.symbol} changed from {e.old_price} to {e.new_price}')

market = StockMarket()
market.subscribe(log_all_changes)
market.subscribe(alert_large_move)

print('Simulating stock price changes:')
market.update_price('AAPL', 178.50)
market.update_price('AAPL', 185.90)   # +4%
market.update_price('TSLA', 260.00)
market.update_price('TSLA', 233.00)   # -10% → triggers alert

# ─── Example 2: Command Pattern — Undo/Redo Text Editor ──────────────────────
from abc import ABC, abstractmethod

class Command(ABC):
    @abstractmethod
    def execute(self): ...
    @abstractmethod
    def undo(self): ...

class Document:
    def __init__(self): self.content = ''
    def __repr__(self): return f'Doc("{self.content}")'

class InsertCommand(Command):
    def __init__(self, doc: Document, pos: int, text: str):
        self.doc, self.pos, self.text = doc, pos, text
    def execute(self): self.doc.content = self.doc.content[:self.pos] + self.text + self.doc.content[self.pos:]
    def undo(self): self.doc.content = self.doc.content[:self.pos] + self.doc.content[self.pos + len(self.text):]

class DeleteCommand(Command):
    def __init__(self, doc: Document, pos: int, length: int):
        self.doc, self.pos, self.length = doc, pos, length
        self._deleted = ''
    def execute(self):
        self._deleted = self.doc.content[self.pos:self.pos + self.length]
        self.doc.content = self.doc.content[:self.pos] + self.doc.content[self.pos + self.length:]
    def undo(self): self.doc.content = self.doc.content[:self.pos] + self._deleted + self.doc.content[self.pos:]

class TextEditor:
    def __init__(self, doc: Document):
        self.doc = doc
        self._history: list[Command] = []
        self._redo_stack: list[Command] = []

    def execute(self, cmd: Command):
        cmd.execute()
        self._history.append(cmd)
        self._redo_stack.clear()  # Any new action clears the redo stack

    def undo(self):
        if self._history:
            cmd = self._history.pop()
            cmd.undo()
            self._redo_stack.append(cmd)

    def redo(self):
        if self._redo_stack:
            cmd = self._redo_stack.pop()
            cmd.execute()
            self._history.append(cmd)

doc = Document()
editor = TextEditor(doc)
editor.execute(InsertCommand(doc, 0, 'Hello'))
editor.execute(InsertCommand(doc, 5, ', World'))
print(f'After inserts: {doc}')
editor.execute(DeleteCommand(doc, 5, 7))
print(f'After delete:  {doc}')
editor.undo()
print(f'After undo:    {doc}')
editor.undo()
print(f'After undo:    {doc}')
editor.redo()
print(f'After redo:    {doc}')

# ─── Example 3: Template Method — Data Pipeline ───────────────────────────────
class DataPipeline(ABC):
    """
    Template Method: defines the pipeline skeleton.
    Subclasses implement the steps; the order is fixed here.
    """
    def run(self):  # Template method — subclasses must NOT override this
        raw = self.extract()
        transformed = self.transform(raw)
        validated = self.validate(transformed)
        self.load(validated)
        print(f'✅ Pipeline {self.__class__.__name__} complete: {len(validated)} records')

    @abstractmethod
    def extract(self) -> list: ...

    @abstractmethod
    def transform(self, data: list) -> list: ...

    def validate(self, data: list) -> list:  # Default: filter falsy rows; subclasses may override
        return [row for row in data if row]

    @abstractmethod
    def load(self, data: list): ...

class CSVToDatabase(DataPipeline):
    def extract(self): return ['  alice,30  ', '  bob,25  ', '', '  carol,35  ']
    def transform(self, data):
        result = []
        for row in data:
            row = row.strip()
            if row:
                name, age = row.split(',')
                result.append({'name': name.strip(), 'age': int(age.strip())})
        return result
    def load(self, data): print(f'  💾 Inserting {len(data)} rows into PostgreSQL: {data}')

class JSONToElasticsearch(DataPipeline):
    def extract(self): return [{'log': 'info msg', 'ts': 1714000000}, None, {'log': 'error!', 'ts': 1714000001}]
    def transform(self, data): return [{'@timestamp': d['ts'], 'message': d['log']} for d in data if d]
    def load(self, data): print(f'  🔍 Indexing {len(data)} documents into Elasticsearch')

CSVToDatabase().run()
JSONToElasticsearch().run()


## 2. Strategy & Command

### Strategy Pattern
**"Define a family of algorithms, encapsulate each one, and make them interchangeable."**

Without Strategy, a sort endpoint might look like `if sort_by == 'price': ... elif sort_by == 'rating': ...`. Every new sort option requires editing that block. With Strategy, each algorithm is a function (or class), stored in a dict or passed as a parameter. Adding a new strategy is adding one new function — no existing code changes.

### Command Pattern
**"Encapsulate a request as an object, thereby allowing you to parameterise clients with different requests, queue or log requests, and support undoable operations."**

The Command pattern is the engineering backbone behind undo/redo, job queues, audit logs, and macro recording. The critical insight is that *actions become data* — they can be stored, replayed, reversed, or sent over a network. Once you understand this pattern, features like "retry failed jobs" or "show history of changes" become straightforward to implement.

In [ ]:
# Strategy — swap algorithms at runtime
from typing import Callable
def price_asc(items): return sorted(items, key=lambda x: x['price'])
def rating_desc(items): return sorted(items, key=lambda x: x['rating'], reverse=True)
def newest_first(items): return sorted(items, key=lambda x: x['created_at'], reverse=True)

class ProductCatalog:
    def __init__(self, sort_strategy: Callable = price_asc):
        self.sort = sort_strategy
    def list(self, products): return self.sort(products)

products = [
    {'name':'Widget A','price':29,'rating':4.2,'created_at':1700000000},
    {'name':'Widget B','price':15,'rating':4.8,'created_at':1710000000},
]
catalog = ProductCatalog(rating_desc)
print('By rating:', [p['name'] for p in catalog.list(products)])

## 3. AI Lab: Pattern Recommendation

### 🧪 AI Lab / Practice

> **TODO:** Describe a module you are currently building in 3-4 sentences → Ask Claude: 'Given this module description, which design pattern(s) do you recommend? Provide a code skeleton with justification.' → Evaluate: is the recommendation appropriate? What would you change?

**Module Description:** Hệ thống cần xây dựng một module xử lý đơn hàng thương mại điện tử theo một quy trình cố định: Xác thực thông tin -> Trừ kho sản phẩm -> Thực hiện thanh toán -> Gửi thông báo cho khách hàng. Trong đó, các bước kiểm tra thông tin và trừ kho là cố định, riêng bước thanh toán phải linh hoạt thay đổi tùy theo phương thức khách hàng chọn (Momo, Thẻ tín dụng, Thẻ ATM)

###Pattern Recommendation:
**Template Method Pattern:** Đóng vai trò làm bộ khung cố định thứ tự 4 bước xử lý đơn hàng ở class cha. Điều này đảm bảo không quên bước gửi thông báo.

**Strategy Pattern:** inject vào ngay tại bước thứ 3 (thực hiện thanh toán) của Template Method nhằm giúp hoán đổi linh hoạt các cổng thanh toán tại runtime.

In [ ]:
from abc import ABC, abstractmethod
from typing import Callable

# ─── STRATEGY PATTERN (Các thuật toán thanh toán) ───────────────────
def pay_via_momo(amount: float):
    print(f"Đang thanh toán {amount} qua Momo...")
    return True

def pay_via_card(amount: float):
    print(f"Đang thanh toán {amount} qua Thẻ...")
    return True

# ─── TEMPLATE METHOD PATTERN (Khung quy trình xử lý đơn) ─────────────
class OrderPipeline(ABC):
    def run_process(self, order_id: str, amount: float, payment_strategy: Callable):
        """Template Method: Định vị chính xác luồng chạy, cấm class con sửa"""
        self._validate_order(order_id)
        self._deduct_stock(order_id)
        
        # Gọi sang Strategy được truyền vào
        success = payment_strategy(amount)
        
        if success:
            self._send_notification(order_id)
            print(f"Hoàn thành xử lý đơn hàng {order_id}\n")

    def _validate_order(self, order_id: str):
        print(f"Xác thực thông tin đơn hàng {order_id}")

    def _deduct_stock(self, order_id: str):
        print(f"Khóa và trừ số lượng sản phẩm trong kho")

    def _send_notification(self, order_id: str):
        print(f"Gửi email xác nhận đơn hàng thành công")

# Class con triển khai luồng cụ thể
class StandardOrderProcessor(OrderPipeline):
    pass # Thừa hưởng toàn bộ khung từ class cha

# Demo
if __name__ == "__main__":
    processor = StandardOrderProcessor()
    # Chạy quy trình với chiến lược Momo
    processor.run_process("ORD-111", 500000, pay_via_momo)

In [ ]:
#rollback kho khi thanh toán thất bại
from abc import ABC, abstractmethod
from typing import Callable

# ─── STRATEGY PATTERN (Các thuật toán thanh toán) ───────────────────
def pay_via_momo(amount: float) -> bool:
    print(f"[Strategy] Đang thanh toán {amount} VNĐ qua Momo...")
    return True

def pay_via_credit_card(amount: float) -> bool:
    print(f"💳 [Strategy] Đang thanh toán {amount} VNĐ qua Thẻ Tín Dụng...")
    # Giả lập tình huống: Khách hàng nhập sai mã OTP hoặc tài khoản không đủ tiền
    return False


# ─── TEMPLATE METHOD PATTERN: Bộ khung quy trình xử lý đơn hàng ─────
class OrderPipeline(ABC):
    
    def run_process(self, order_id: str, amount: float, payment_strategy: Callable[[float], bool]):
        """
        Template Method: Định nghĩa bộ khung quy trình nghiêm ngặt.
        Có tích hợp cơ chế tự động Rollback kho nếu xảy ra sự cố ở bước sau.
        """
        print(f"🏁 [Quy trình] Bắt đầu xử lý đơn hàng: {order_id}")
        
        self._validate_order(order_id)
        
        # Bước trừ kho bắt buộc phải chạy trước thanh toán
        self._deduct_stock(order_id)
        
        # Đặt quá trình thanh toán vào cơ chế kiểm soát an toàn
        try:
            payment_success = payment_strategy(amount)
            
            if payment_success:
                print("✅ Thanh toán thành công!")
                self._send_notification(order_id)
                print(f"🎉 [Quy trình] Đơn hàng {order_id} đã hoàn tất trọn vẹn.\n")
            else:
                print("❌ Thanh toán thất bại hoặc bị hủy bởi người dùng.")
                # THANH TOÁN THẤT BẠI -> Tự động kích hoạt cơ chế hoàn tác nhả kho
                self._rollback_stock(order_id)
                print(f"↩️ [Quy trình] Đã hủy đơn {order_id} an toàn, không lo kẹt kho.\n")
                
        except Exception as e:
            # PHÁT SINH LỖI HỆ THỐNG (Mất mạng, sập cổng thanh toán...) -> Vẫn phải rollback kho
            print(f"🚨 Phát sinh lỗi hệ thống nghiêm trọng trong lúc thanh toán: {e}")
            self._rollback_stock(order_id)
            print(f"↩️ [Quy trình] Đã kích hoạt khẩn cấp cơ chế Rollback cho đơn {order_id}.\n")

    def _validate_order(self, order_id: str):
        print(f"🔍 Bước 1: Xác thực thông tin đơn hàng {order_id} hợp lệ.")

    def _deduct_stock(self, order_id: str):
        print(f"📦 Bước 2: Giảm số lượng trong Database kho (-1 sản phẩm cho đơn {order_id}).")

    def _rollback_stock(self, order_id: str):
        """Hàm hoàn tác (Rollback): Trả lại trạng thái cũ cho kho dữ liệu nếu có sự cố xảy ra"""
        print(f"🔄 [Rollback Kho]: Khôi phục dữ liệu! Cộng trả lại (+1 sản phẩm) vào kho cho đơn {order_id}.")

    @abstractmethod
    def _send_notification(self, order_id: str):
        """Để class con tự quyết định kênh gửi thông báo (Email, SMS hoặc Zalo)"""
        pass


# ─── CLASS CON TRIỂN KHAI CHI TIẾT ─────────────────────────────────
class StandardOrderProcessor(OrderPipeline):
    def _send_notification(self, order_id: str):
        print(f"📧 Bước 4: Gửi email hóa đơn điện tử cho khách hàng đặt đơn {order_id}.")


# ─── CHẠY MÔ PHỎNG THỰC TẾ (DEMO) ──────────────────────────────────
if __name__ == "__main__":
    processor = StandardOrderProcessor()

    # KỊCH BẢN 1: Khách hàng chọn Momo và thanh toán thành công ngon lành
    print("=== KỊCH BẢN 1: THANH TOÁN THÀNH CÔNG ===")
    processor.run_process(
        order_id="ORD-SUCCESS-123", 
        amount=250000, 
        payment_strategy=pay_via_momo
    )

    # KỊCH BẢN 2: Khách hàng chọn Thẻ tín dụng nhưng thanh toán thất bại (Kích hoạt Rollback)
    print("=== KỊCH BẢN 2: THANH TOÁN THẤT BẠI -> TỰ ĐỘNG ROLLBACK ===")
    processor.run_process(
        order_id="ORD-FAILED-456", 
        amount=500000, 
        payment_strategy=pay_via_credit_card
    )